<a href="https://colab.research.google.com/github/SebMay99/-MAGIC-Gamma-Telescope---ML-Classification-Models/blob/main/MAGIC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler

# ── create output folder for all saved figures ──────────────────────────────
os.makedirs("figures", exist_ok=True)

In [ ]:
cols = ["fLength", "fWidth", "fSize", "fConc", "fConc1", "fAsyn","fM3Long","fM3Trans", "fAlpha", "fDist", "class"]
df = pd.read_csv("magic04.data", names=cols)
df.head()

In [ ]:
print(df["class"].unique())

In [ ]:
df["class"] = (df["class"] == "g").astype(int)
df.head()

# EDA

In [ ]:
feature_cols = cols[:-1]

# ── 1. Class balance bar chart ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
counts = df["class"].value_counts().sort_index()
ax.bar(["Hadron (0)", "Gamma (1)"], counts.values, color=["#e05c5c", "#4c8be0"])
ax.set_title("Class Distribution", fontsize=13, fontweight="bold")
ax.set_ylabel("Count")
for i, v in enumerate(counts.values):
    ax.text(i, v + 80, f"{v:,}", ha="center", fontsize=10)
plt.tight_layout()
plt.savefig("figures/eda_class_balance.png", dpi=150)
plt.show()
print("Saved: figures/eda_class_balance.png")

In [ ]:
# ── 2. Feature histograms grid (all 10 features, gamma vs hadron) ────────────
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
axes = axes.flatten()
for i, label in enumerate(feature_cols):
    ax = axes[i]
    ax.hist(df[df["class"] == 1][label], color="#4c8be0", label="Gamma", alpha=0.7, density=True, bins=40)
    ax.hist(df[df["class"] == 0][label], color="#e05c5c", label="Hadron", alpha=0.7, density=True, bins=40)
    ax.set_title(label, fontsize=10, fontweight="bold")
    ax.set_ylabel("Density" if i % 5 == 0 else "")
    ax.set_xlabel(label)
    if i == 0:
        ax.legend(fontsize=8)
plt.suptitle("Feature Distributions: Gamma vs Hadron", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("figures/eda_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/eda_feature_distributions.png")

In [ ]:
# ── 3. Correlation heatmap ───────────────────────────────────────────────────
import matplotlib.colors as mcolors

corr = df[feature_cols].corr()
fig, ax = plt.subplots(figsize=(9, 7))
cmap = plt.cm.coolwarm
im = ax.imshow(corr, cmap=cmap, vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(len(feature_cols)))
ax.set_yticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(feature_cols, fontsize=9)
for i in range(len(feature_cols)):
    for j in range(len(feature_cols)):
        val = corr.iloc[i, j]
        color = "white" if abs(val) > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7, color=color)
ax.set_title("Feature Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/eda_correlation_heatmap.png", dpi=150)
plt.show()
print("Saved: figures/eda_correlation_heatmap.png")

# Train, validation, test datasets

In [ ]:
train, valid, test = np.split(df.sample(frac=1, random_state=42), [int(0.6*len(df)), int(0.8*len(df))])

In [ ]:
def scale_dataset(dataframe, oversample=False):
    X = dataframe[dataframe.columns[:-1]].values
    y = dataframe[dataframe.columns[-1]].values
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    if oversample:
        ros = RandomOverSampler()
        X, y = ros.fit_resample(X, y)
    data = np.hstack((X, np.reshape(y, (-1,1))))
    return data, X, y

In [ ]:
train, X_train, y_train = scale_dataset(train, oversample=True)
valid, X_valid, y_valid = scale_dataset(valid, oversample=False)
test,  X_test,  y_test  = scale_dataset(test,  oversample=False)

# Helper: plot & save confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

def save_confusion_matrix(y_true, y_pred, model_name, filename):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Hadron", "Gamma"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"Confusion Matrix — {model_name}", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"figures/{filename}", dpi=150)
    plt.show()
    print(f"Saved: figures/{filename}")

# k-Nearest Neighbors

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)
print(classification_report(y_test, y_pred_knn))
save_confusion_matrix(y_test, y_pred_knn, "kNN (k=5)", "cm_knn.png")

# Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)
print(classification_report(y_test, y_pred_nb))
save_confusion_matrix(y_test, y_pred_nb, "Naive Bayes", "cm_naive_bayes.png")

# Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
log_model = LogisticRegression()
log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)
print(classification_report(y_test, y_pred_log))
save_confusion_matrix(y_test, y_pred_log, "Logistic Regression", "cm_logistic_regression.png")

# SVM

In [ ]:
from sklearn.svm import SVC
svm_model = SVC()
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)
print(classification_report(y_test, y_pred_svm))
save_confusion_matrix(y_test, y_pred_svm, "SVM (RBF)", "cm_svm.png")

# Neural Network

In [ ]:
import tensorflow as tf

In [ ]:
def train_model(X_train, y_train, num_nodes, dropout_prob, lr, batch_size, epochs):
    nn_model = tf.keras.Sequential([
        tf.keras.layers.Dense(num_nodes, activation='relu', input_shape=(10,)),
        tf.keras.layers.Dropout(dropout_prob),
        tf.keras.layers.Dense(num_nodes, activation='relu'),
        tf.keras.layers.Dropout(dropout_prob),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    nn_model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                     loss='binary_crossentropy', metrics=['accuracy'])
    history = nn_model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size,
                           validation_data=(X_valid, y_valid), verbose=0)
    return nn_model, history

In [ ]:
least_val_loss = float('inf')
least_loss_model = None
best_history = None
best_params = {}
epochs = 100

for num_nodes in [16, 32, 64]:
    for dropout_prob in [0, 0.2]:
        for lr in [0.1, 0.005, 0.001]:
            for batch_size in [32, 64, 128]:
                print(f"{num_nodes} nodes, dropout {dropout_prob}, lr {lr}, batch_size {batch_size}")
                model, history = train_model(X_train, y_train, num_nodes, dropout_prob, lr, batch_size, epochs)
                val_loss = model.evaluate(X_valid, y_valid, verbose=0)[0]
                if val_loss < least_val_loss:
                    least_val_loss = val_loss
                    least_loss_model = model
                    best_history = history
                    best_params = dict(num_nodes=num_nodes, dropout=dropout_prob, lr=lr, batch_size=batch_size)

print("\nBest params:", best_params)
print(f"Best val loss: {least_val_loss:.4f}")

In [ ]:
# ── Training curves for BEST model only ─────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(best_history.history['loss'],     label='Train loss',  color='#4c8be0')
ax1.plot(best_history.history['val_loss'], label='Val loss',    color='#e05c5c')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Binary Cross-Entropy')
ax1.set_title('Loss Curves — Best Neural Network', fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(best_history.history['accuracy'],     label='Train acc',  color='#4c8be0')
ax2.plot(best_history.history['val_accuracy'], label='Val acc',    color='#e05c5c')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy Curves — Best Neural Network', fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)

label_str = (f"nodes={best_params['num_nodes']}, dropout={best_params['dropout']}, "
             f"lr={best_params['lr']}, batch={best_params['batch_size']}")
fig.suptitle(f"Best config: {label_str}", fontsize=10, y=1.01)
plt.tight_layout()
plt.savefig("figures/nn_best_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/nn_best_training_curves.png")

In [ ]:
y_pred_nn = least_loss_model.predict(X_test)
y_pred_nn = (y_pred_nn > 0.5).astype(int).reshape(-1,)
print(classification_report(y_test, y_pred_nn))
save_confusion_matrix(y_test, y_pred_nn, "Neural Network (best)", "cm_neural_network.png")

# Model Comparison

In [ ]:
from sklearn.metrics import roc_curve, auc, accuracy_score

# ── ROC curve comparison ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

roc_models = {
    "kNN":                 (knn_model, "predict_proba"),
    "Naive Bayes":         (nb_model,  "predict_proba"),
    "Logistic Regression": (log_model, "predict_proba"),
    "SVM":                 (svm_model, "decision_function"),
    "Neural Network":      (least_loss_model, "nn"),
}
colors = ["#4c8be0", "#e07c4c", "#4cba6e", "#c44ce0", "#e05c5c"]

for (name, (mdl, kind)), color in zip(roc_models.items(), colors):
    if kind == "predict_proba":
        scores = mdl.predict_proba(X_test)[:, 1]
    elif kind == "decision_function":
        scores = mdl.decision_function(X_test)
    else:
        scores = mdl.predict(X_test).flatten()
    fpr, tpr, _ = roc_curve(y_test, scores)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC = {roc_auc:.3f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title("ROC Curves — All Models", fontsize=13, fontweight="bold")
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/comparison_roc_curves.png", dpi=150)
plt.show()
print("Saved: figures/comparison_roc_curves.png")

In [ ]:
# ── Accuracy & F1 bar chart comparison ──────────────────────────────────────
from sklearn.metrics import f1_score

model_names  = ["Naive Bayes", "Logistic Reg", "kNN", "SVM", "Neural Net"]
y_preds      = [y_pred_nb, y_pred_log, y_pred_knn, y_pred_svm, y_pred_nn]
accuracies   = [accuracy_score(y_test, yp) for yp in y_preds]
f1_gamma     = [f1_score(y_test, yp, pos_label=1) for yp in y_preds]
f1_hadron    = [f1_score(y_test, yp, pos_label=0) for yp in y_preds]

x = np.arange(len(model_names))
w = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w,   accuracies, w, label="Accuracy",   color="#4c8be0")
b2 = ax.bar(x,       f1_gamma,   w, label="F1 Gamma",   color="#4cba6e")
b3 = ax.bar(x + w,   f1_hadron,  w, label="F1 Hadron",  color="#e05c5c")

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel("Score", fontsize=11)
ax.set_title("Model Comparison: Accuracy & F1 Scores", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=7.5)

plt.tight_layout()
plt.savefig("figures/comparison_accuracy_f1.png", dpi=150)
plt.show()
print("Saved: figures/comparison_accuracy_f1.png")

In [ ]:
print("\n=== All figures saved to ./figures/ ===")
import os
for f in sorted(os.listdir("figures")):
    print(f"  figures/{f}")